# TBP Data Mining — Anggota 1 (Laura)
## Modeling Mandiri: Decision Tree (CART / Gini)

**Dataset Bersama:** Early Warning System Kelulusan Mahasiswa (`output/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`)

### Aturan Tim yang Dipatuhi
- Menggunakan file di folder `output/` apa adanya (**tanpa split ulang / acak ulang**).
- **Tidak** menggunakan `scaler.pkl` (Decision Tree berbasis aturan pembagian, tidak membutuhkan scaling).
- Tuning hyperparameter (`max_depth`) dilakukan **hanya pada `X_train`** (5-Fold Stratified CV) tanpa menyentuh `X_test`.
- Menyimpan luaran lengkap untuk Fase 3 (setoran ke Agischa/Gisca) & laporan di folder `hasil_laura_dt/`:
  - `decision_tree_model.pkl`
  - `prediksi_decision_tree.csv` (`y_true`, `y_pred`, `y_proba` untuk kelas `1` = Graduate)
  - `metrik_decision_tree.json`, `tabel_tuning_max_depth.csv`, `feature_importance_dt.csv`
  - `tree_plot_decision_tree.png` & `confusion_matrix_roc_dt.png`
  - `aturan_if_then_decision_tree.txt`

In [ ]:
from pathlib import Path
import hashlib
import json
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, _tree, export_text, plot_tree

# Jika dijalankan di Google Colab dan folder `output/` ada di Google Drive,
# buka komentar 2 baris berikut lalu sesuaikan DATA_DIR:
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = Path("output")
OUT_DIR = Path("hasil_laura_dt")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_MD5 = {
    "X_train.csv": "a082607c430d44f0dcb3993748f8c6a6",
    "X_test.csv": "4127b1a07685e0f73a400e51bdafb12c",
    "y_train.csv": "f3d30dd28c1a8470aa98b1b58b34f6f1",
    "y_test.csv": "67030176445e89021fc6ee58ee17bbbe",
}

RANDOM_STATE = 42
CANDIDATE_MAX_DEPTHS = [3, 4, 5, 7, 10, None]

## 1. Verifikasi Integritas File (MD5) & Muat Dataset Bersama
Memastikan file CSV di folder `output/` sama persis dengan file resmi tim sebelum modeling.

In [ ]:
def hitung_md5(filepath: Path) -> str:
    hasher = hashlib.md5()
    with filepath.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


print("=== VERIFIKASI MD5 DATASET BERSAMA ===")
for filename, expected in EXPECTED_MD5.items():
    fpath = DATA_DIR / filename
    if not fpath.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {fpath.resolve()}\n"
            "Pastikan folder `output/` berada di lokasi DATA_DIR."
        )
    actual = hitung_md5(fpath)
    status = "OK" if actual == expected else "BERBEDA (cek versi/line-ending)"
    print(f"  - {filename:<12} | MD5: {actual} | Status: {status}")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(f"\nDimensi X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"Dimensi X_test : {X_test.shape}  | y_test : {y_test.shape}")
print("Distribusi y_train:", y_train.value_counts().sort_index().to_dict())
print("Distribusi y_test :", y_test.value_counts().sort_index().to_dict())
X_train.head()

## 2. Tuning `max_depth` Decision Tree (CART / Gini) di `X_train`
Menguji beberapa batas kedalaman pohon dengan **5-Fold Stratified Cross-Validation** hanya pada `X_train` untuk membandingkan kecenderungan *overfitting* (pohon `None` vs pohon ter-*pruning*) sekaligus memilih `max_depth` terbaik.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
hasil_tuning = []

for depth in CANDIDATE_MAX_DEPTHS:
    cv_acc, cv_rec0, cv_f1_0, cv_auc = [], [], [], []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        clf = DecisionTreeClassifier(
            criterion="gini",
            max_depth=depth,
            random_state=RANDOM_STATE,
        )
        clf.fit(X_tr, y_tr)
        val_pred = clf.predict(X_val)
        val_proba = clf.predict_proba(X_val)[:, 1]

        cv_acc.append(accuracy_score(y_val, val_pred))
        cv_rec0.append(recall_score(y_val, val_pred, pos_label=0))
        cv_f1_0.append(f1_score(y_val, val_pred, pos_label=0))
        cv_auc.append(roc_auc_score(y_val, val_proba))

    # Cek akurasi pada seluruh X_train untuk mengamati overfitting
    clf_full = DecisionTreeClassifier(
        criterion="gini",
        max_depth=depth,
        random_state=RANDOM_STATE,
    )
    clf_full.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, clf_full.predict(X_train))

    hasil_tuning.append(
        {
            "max_depth": "None (Unpruned)" if depth is None else str(depth),
            "depth_value": depth,
            "train_accuracy": round(float(train_acc), 4),
            "cv_accuracy_mean": round(float(np.mean(cv_acc)), 4),
            "cv_recall_dropout_0_mean": round(float(np.mean(cv_rec0)), 4),
            "cv_f1_dropout_0_mean": round(float(np.mean(cv_f1_0)), 4),
            "cv_roc_auc_mean": round(float(np.mean(cv_auc)), 4),
            "n_leaves_full_train": int(clf_full.get_n_leaves()),
        }
    )

df_tuning = pd.DataFrame(hasil_tuning)
df_tuning.drop(columns=["depth_value"]).to_csv(
    OUT_DIR / "tabel_tuning_max_depth.csv", index=False
)

kandidat_pruned = df_tuning[df_tuning["depth_value"].notna()].sort_values(
    by=["cv_roc_auc_mean", "cv_recall_dropout_0_mean", "depth_value"],
    ascending=[False, False, True],
)
best_depth = int(kandidat_pruned.iloc[0]["depth_value"])

print(f"Dipilih max_depth = {best_depth} (terbaik di antara kandidat ter-pruning pada 5-Fold CV).")
df_tuning.drop(columns=["depth_value"])

## 3. Training Model Final & Penyimpanan `.pkl`
Melatih model Decision Tree (CART) dengan `max_depth` terpilih pada seluruh `X_train`, mencatat waktu training, dan menyimpan model ke `decision_tree_model.pkl`.

In [ ]:
final_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=best_depth,
    random_state=RANDOM_STATE,
)

t_start = time.perf_counter()
final_model.fit(X_train, y_train)
training_time_sec = time.perf_counter() - t_start

model_path = OUT_DIR / "decision_tree_model.pkl"
with model_path.open("wb") as f:
    pickle.dump(final_model, f)

print(f"Model tersimpan di : {model_path}")
print(f"Waktu training     : {training_time_sec:.4f} detik")
print(f"Kedalaman aktual   : {final_model.get_depth()} | Jumlah daun (leaves): {final_model.get_n_leaves()}")

## 4. Evaluasi pada `X_test` & Simpan Paket Setoran Fase 3
Menghitung Accuracy, Precision, **Recall (khususnya kelas Dropout = `0`)**, F1-Score, Confusion Matrix, dan ROC-AUC, lalu menyimpan `y_pred`, `y_proba` (probabilitas kelas `1` = Graduate), metrik JSON, dan Feature Importance.

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]  # Probabilitas kelas 1 (Graduate)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn_d, fp_d, fn_d, tp_d = cm.ravel()

metrics = {
    "model_name": "Decision Tree (CART / Gini)",
    "anggota": "Anggota 1 - Laura",
    "params": {
        "criterion": "gini",
        "max_depth": best_depth,
        "random_state": RANDOM_STATE,
    },
    "training_time_sec": round(float(training_time_sec), 4),
    "n_leaves": int(final_model.get_n_leaves()),
    "tree_depth_actual": int(final_model.get_depth()),
    "test_accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "test_roc_auc": round(float(roc_auc_score(y_test, y_proba)), 4),
    "test_precision_dropout_0": round(float(precision_score(y_test, y_pred, pos_label=0)), 4),
    "test_recall_dropout_0": round(float(recall_score(y_test, y_pred, pos_label=0)), 4),
    "test_f1_dropout_0": round(float(f1_score(y_test, y_pred, pos_label=0)), 4),
    "test_precision_graduate_1": round(float(precision_score(y_test, y_pred, pos_label=1)), 4),
    "test_recall_graduate_1": round(float(recall_score(y_test, y_pred, pos_label=1)), 4),
    "test_f1_graduate_1": round(float(f1_score(y_test, y_pred, pos_label=1)), 4),
    "test_precision_macro": round(float(precision_score(y_test, y_pred, average="macro")), 4),
    "test_recall_macro": round(float(recall_score(y_test, y_pred, average="macro")), 4),
    "test_f1_macro": round(float(f1_score(y_test, y_pred, average="macro")), 4),
    "confusion_matrix_labels_0_1": {
        "actual_0_pred_0_correct_dropout": int(tn_d),
        "actual_0_pred_1_missed_dropout": int(fp_d),
        "actual_1_pred_0_false_alarm": int(fn_d),
        "actual_1_pred_1_correct_graduate": int(tp_d),
    },
}

print(classification_report(y_test, y_pred, target_names=["Dropout (0)", "Graduate (1)"], digits=4))
print(f"Accuracy          : {metrics['test_accuracy']:.4f}")
print(f"Recall Dropout (0): {metrics['test_recall_dropout_0']:.4f}  <-- Metrik prioritas EWS")
print(f"ROC-AUC           : {metrics['test_roc_auc']:.4f}")
print(f"Waktu Training    : {metrics['training_time_sec']:.4f} detik")

# Simpan prediksi & metrik untuk evaluasi bersama (Anggota 3)
pd.DataFrame({"y_true": y_test.values, "y_pred": y_pred, "y_proba": y_proba}).to_csv(
    OUT_DIR / "prediksi_decision_tree.csv", index=False
)
with (OUT_DIR / "metrik_decision_tree.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

df_fi = pd.DataFrame(
    {"Fitur": X_train.columns, "Importance": final_model.feature_importances_}
).sort_values(by="Importance", ascending=False)
df_fi.to_csv(OUT_DIR / "feature_importance_dt.csv", index=False)
df_fi

## 5. Visualisasi Diagram Pohon Keputusan (*Tree Plot*), Confusion Matrix, & Kurva ROC
Menampilkan visualisasi langsung di notebook sekaligus menyimpannya dalam resolusi 300 DPI untuk laporan dan slide presentasi.

In [ ]:
# 5a. Diagram Pohon Keputusan (Tree Plot)
plot_depth = min(best_depth, 3)
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    final_model,
    max_depth=plot_depth,
    feature_names=list(X_train.columns),
    class_names=["Dropout (0)", "Graduate (1)"],
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=9,
    ax=ax,
)
ax.set_title(
    f"Visualisasi Decision Tree (CART) — max_depth={best_depth} (Ditampilkan s.d. level {plot_depth})",
    fontsize=14,
    fontweight="bold",
    pad=14,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "tree_plot_decision_tree.png", dpi=300, bbox_inches="tight")
plt.show()

# 5b. Confusion Matrix & Kurva ROC
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Dropout (0)", "Graduate (1)"],
).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
axes[0].set_title("Confusion Matrix — Decision Tree (Test Set)", fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#1f77b4", lw=2, label=f"Decision Tree (AUC = {metrics['test_roc_auc']:.4f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1.2, linestyle="--", label="Baseline Acak (AUC = 0.50)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Kurva ROC — Decision Tree (Test Set)", fontweight="bold")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix_roc_dt.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Ekstraksi Aturan *If-Then*
Mengekstrak struktur pohon (`export_text`) serta daftar lengkap aturan `IF ... THEN ...` per daun (leaf) ke `hasil_laura_dt/aturan_if_then_decision_tree.txt`.

In [ ]:
def ekstrak_aturan_if_then(model: DecisionTreeClassifier, feature_names: list[str]) -> list[str]:
    tree_ = model.tree_
    feature_name = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!"
        for i in tree_.feature
    ]
    aturan = []

    def telusuri(node: int, kondisi: list[str]) -> None:
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            nama_fitur = feature_name[node]
            ambang = tree_.threshold[node]
            telusuri(tree_.children_left[node], kondisi + [f"{nama_fitur} <= {ambang:.2f}"])
            telusuri(tree_.children_right[node], kondisi + [f"{nama_fitur} > {ambang:.2f}"])
        else:
            counts = tree_.value[node][0]
            total_sampel = int(tree_.n_node_samples[node])
            prob_dropout = counts[0] / counts.sum()
            prob_graduate = counts[1] / counts.sum()
            pred_kelas = "Graduate (1)" if prob_graduate >= prob_dropout else "Dropout (0)"
            if_clause = " AND ".join(kondisi) if kondisi else "(Root)"
            aturan.append(
                f"IF {if_clause} THEN {pred_kelas} "
                f"[sampel={total_sampel}, P(Dropout)={prob_dropout:.3f}, P(Graduate)={prob_graduate:.3f}]"
            )

    telusuri(0, [])
    return aturan


teks_pohon = export_text(final_model, feature_names=list(X_train.columns), decimals=2)
daftar_aturan = ekstrak_aturan_if_then(final_model, list(X_train.columns))

aturan_path = OUT_DIR / "aturan_if_then_decision_tree.txt"
with aturan_path.open("w", encoding="utf-8") as f:
    f.write("=====================================================================\n")
    f.write(f"EKSTRAKSI ATURAN DECISION TREE (CART) - max_depth={best_depth}\n")
    f.write("Catatan: Pekerjaan Sambil Kuliah (0=Tidak, 1=Ya) | ")
    f.write("Kategori Kehadiran (0=Rendah, 1=Sedang, 2=Tinggi)\n")
    f.write("=====================================================================\n\n")
    f.write("1. STRUKTUR POHON (export_text)\n")
    f.write("---------------------------------------------------------------------\n")
    f.write(teks_pohon)
    f.write("\n2. DAFTAR ATURAN IF-THEN PER DAUN (LEAF)\n")
    f.write("---------------------------------------------------------------------\n")
    for idx, rule in enumerate(daftar_aturan, start=1):
        f.write(f"{idx:02d}. {rule}\n")

print(teks_pohon)
print(f"Total aturan IF-THEN diekstrak: {len(daftar_aturan)} (disimpan ke {aturan_path})")
print("\nDaftar seluruh file luaran di folder hasil_laura_dt/:")
for item in sorted(OUT_DIR.iterdir()):
    print(f"  - {item.name}")